## Run test: Qwen/Qwen2.5-VL-3B-Instruct

Aligned with the Hugging Face README Quickstart. Inference Providers snippets are ignored because they do not validate local AMD GPU loading. The direct path uses Qwen2_5_VLForConditionalGeneration, matching the README.

In [ ]:
# README dependency for image/video input preprocessing.
%pip install -U transformers accelerate "qwen-vl-utils[decord]"

In [ ]:
# Pipeline smoke test on one visible GPU, then release memory before direct loading.
import gc
import torch
from transformers import pipeline

model_path = "/disk/ssd1/zihaomu_amd/models/Qwen__Qwen2.5-VL-3B-Instruct"
messages = [{"role": "user", "content": [{"type": "text", "text": "Who are you?"}]}]

pipe = pipeline("image-text-to-text", model=model_path, device=0, trust_remote_code=True)
pipe_result = pipe(messages, max_new_tokens=64)
print(pipe_result)
pipe_tensor_devices = sorted({str(p.device) for p in pipe.model.parameters()})
print("pipe parameter devices =", pipe_tensor_devices)

del pipe
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# README-aligned direct model load smoke test.
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

processor = AutoProcessor.from_pretrained(model_path)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_path).to("cuda").eval()

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"},
            {"type": "text", "text": "Describe this image briefly."},
        ],
    }
]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to("cuda")

with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=64)
trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
output_text = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
print(output_text[0])